# Diffusion models

_Modern Deep Learning & AI_

**Destroy a picture with noise step by step, then teach a network to undo it. Run the undo from pure noise to create.**

A diffusion model learns to generate by first learning to destroy, then reversing it.

     The forward process takes a clean image and adds a little Gaussian noise, again and again, until it is pure static. This part is fixed, no learning needed.

---

This notebook is a practice scaffold for the **Diffusion models** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build one training step of a diffusion model in three parts: (1) set up the fixed noise schedule, (2) noise a clean batch to a random timestep using the closed-form forward jump, (3) train a tiny network to predict the noise that was added. The model never learns to *destroy* — that part is fixed math; it only learns to *name the noise* so it can later undo it.

### Step 1 — Set up the noise schedule and the network

`betas` is the per-step amount of noise across `T` steps. `alpha_bar` is the running product of `(1 - beta)`, which tells us how much of the original signal survives by step `t`. The `net` is a tiny MLP that takes a noised sample plus its timestep and predicts the noise.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 200
betas = torch.linspace(1e-4, 0.02, T)          # noise schedule
alpha_bar = torch.cumprod(1.0 - betas, dim=0)  # product of (1 - beta) up to t

# tiny noise-predictor: takes x_t and a (scaled) timestep, predicts the noise
net = nn.Sequential(
    nn.Linear(5 + 1, 32),
    nn.ReLU(),
    nn.Linear(32, 5),
)

### Step 2 — Noise a clean batch to a random timestep

For each sample we pick a random timestep `t` and look up how much signal survives there (`ab = alpha_bar[t]`). The forward process has a **closed form**: we can jump straight to `x_t` in one shot as `sqrt(ab)*x0 + sqrt(1-ab)*eps`, no step-by-step looping needed. `eps` is the exact noise we mixed in — that's the answer the network must learn to recover.

In [ ]:
torch.manual_seed(0)
x0 = torch.rand(8, 5)                           # clean synthetic data
t = torch.randint(0, T, (8,))                   # a random step per sample
ab = alpha_bar[t].unsqueeze(1)                  # (8, 1) signal survival at step t

eps = torch.randn_like(x0)                      # the noise we add
x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * eps    # closed-form forward jump

### Step 3 — Predict the noise and backprop

We hand the network the noised sample together with its timestep (scaled to 0..1 so it is a sensible feature) and ask it to predict `eps`. The MSE between the predicted and the true noise is the training loss; `backward()` then teaches the network to name the noise — the skill it needs to reverse the process at generation time.

In [ ]:
t_feat = (t.float() / T).unsqueeze(1)           # timestep as a feature
net_input = torch.cat([x_t, t_feat], dim=1)     # noised sample + its timestep
eps_pred = net(net_input)                       # predict the added noise

loss = F.mse_loss(eps_pred, eps)                # train to name the noise
loss.backward()
print("x_t shape:", x_t.shape, "loss:", round(loss.item(), 4))

## Visualize the data & results

_Apply the real DDPM noise schedule to one real digit-3 image: how fast does signal turn into noise?_

We do it in three steps: (1) load one real digit and build the standard 1000-step schedule, (2) noise the image at a spread of timesteps and measure the noise fraction, (3) plot signal vs noise vs the measured fraction to see the image dissolve.

### Step 1 — Load one digit and build the DDPM schedule

We grab one real handwritten "3" and scale it to 0..1. Then we build the standard 1000-step linear schedule and compute, at the timesteps we will sample, how much **signal** (`sqrt(alpha_bar)`) and how much **noise** (`sqrt(1-alpha_bar)`) the image carries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# forward diffusion of a REAL digit image with the standard DDPM schedule
digits = load_digits()
three_idx = np.where(digits.target == 3)[0][0]   # index of the first real "3"
img = (digits.data / 16.0)[three_idx]            # one real "3", scaled 0..1

T = 1000
betas = np.linspace(1e-4, 0.02, T)              # linear noise schedule
alpha_bar = np.cumprod(1.0 - betas)

ts = np.arange(0, T, 100)                        # the timesteps we sample
signal = np.sqrt(alpha_bar[ts])                  # how much original survives
noise = np.sqrt(1.0 - alpha_bar[ts])             # how much noise has crept in

### Step 2 — Noise the image and measure the noise fraction

Using one fixed noise sample `eps`, we form the noised image `xt` at each timestep with the same closed-form jump as before. At each step we measure what fraction of `xt`'s variance comes from the noise term — an empirical check that should track `sqrt(1-alpha_bar)`.

In [ ]:
rng = np.random.default_rng(3)
eps = rng.standard_normal(img.shape)             # one fixed noise sample

meas = []
for t in ts:
    noise_term = np.sqrt(1 - alpha_bar[t]) * eps
    xt = np.sqrt(alpha_bar[t]) * img + noise_term  # noised image at step t
    noise_fraction = np.var(noise_term) / (np.var(xt) + 1e-9)
    meas.append(noise_fraction)

### Step 3 — Plot signal vs noise over time

We draw the surviving signal (blue) falling, the injected noise (red) rising, and our measured noise fraction (orange) tracking the theoretical curve. The crossover is where the digit stops being recognizable and becomes mostly static.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts, signal, color="#4ea1ff", label="signal sqrt(alpha_bar)")
ax.plot(ts, noise, color="#ff7b72", label="noise sqrt(1-alpha_bar)")
ax.plot(ts, np.clip(meas, 0, 1), color="#ffb454", label="measured noise fraction")
ax.set_xlabel("timestep t")
ax.set_ylabel("fraction")
ax.set_title("real digit image dissolving into noise")
ax.legend()
plt.show()